# O que é um resultado?

**Nível 1 — Iniciante** · Trilha Wizz Lab

> Retorno é o que aconteceu. Risco é o que poderia ter acontecido e quase aconteceu.


Este notebook percorre as 26 métricas do nível iniciante. Nenhuma delas usa estatística
inferencial — é aritmética honesta.

A ideia que precisa ficar ao final: **retorno sozinho não é resultado.**


---

In [ ]:
%matplotlib inline

# No Colab, instala o pacote direto do GitHub. Localmente, não faz nada.
import importlib.util, subprocess, sys

if importlib.util.find_spec("wizzlab") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/Gustavofthiesen/wizz-lab.git"], check=True)

from wizzlab import brand, data, metrics, scorecard
from wizzlab.theme import aplicar_tema
from wizzlab import charts

aplicar_tema()
print("Wizz Lab pronto · paleta:", brand.SERIE_PRINCIPAL, brand.SERIE_COMPARACAO)

## 1. Uma estratégia para estudar

Todos os dados aqui são simulados, e isso é uma vantagem pedagógica, não uma limitação:
como nós escolhemos o processo gerador, sabemos qual é a resposta certa e podemos
conferir se a métrica de fato a encontra.

Por construção, esta estratégia tem **win rate de 42% e payoff de 1,9**.

In [ ]:
est = data.gerar_estrategia(semente=42)
r = est.trades.r_multiple.values      # resultado de cada trade, em R
est

## 2. Win rate: a métrica mais mal usada do mercado

Win rate mede **frequência**, não qualidade. Uma estratégia pode ganhar dinheiro
acertando 30% das vezes, e perder acertando 80%.

O que decide não é o acerto isolado — é o acerto **comparado ao payoff**.

In [ ]:
wr = metrics.trade.win_rate(r)
payoff = metrics.trade.payoff_ratio(r)
be = metrics.trade.break_even_win_rate(r)

print(f"Win rate observado ....... {wr:.1%}")
print(f"Payoff (ganho/perda) ..... {payoff:.2f}x")
print(f"Win rate de equilíbrio ... {be:.1%}")
print(f"Margem de edge ........... {wr - be:+.1%}")

Acertar 42% das vezes parece ruim. Com payoff de 1,4x, o equilíbrio está em ~41% — então
42% é apenas *um pouco* melhor que empatar.

**Essa margem de poucos pontos percentuais é exatamente a que desaparece com custos.**
Guarde este número: vamos voltar a ele no nível 2.

In [ ]:
metrics.trade.resumo(r).round(3)

## 3. Média e mediana: quem sustenta o resultado?

A distância entre média e mediana é um diagnóstico direto. Se a média é muito maior que a
mediana, poucos trades excepcionais estão carregando o conjunto.

In [ ]:
fig, ax = charts.distribuicao_trades(r, fonte="Dados simulados · wizz-lab")
fig

A mediana é negativa e a média é positiva. Isso **não é necessariamente ruim**: é o perfil
clássico de um sistema que corta perdas e deixa ganhos correrem. Mas é uma escolha, e
precisa ser consciente — significa depender da cauda direita.

## 4. O preço que a estratégia cobrou

Agora a outra metade do resultado. A curva de capital mostra o destino; o drawdown mostra
o caminho.

In [ ]:
bench = data.gerar_benchmark(est)
fig, ax = charts.curva_capital(est.diario, bench,
                               fonte="Dados simulados · wizz-lab")
fig

In [ ]:
fig, ax = charts.drawdown(est.diario, fonte="Dados simulados · wizz-lab")
fig

In [ ]:
print(f"Queda máxima ............. {metrics.risco.max_drawdown(est.diario):.1%}")
print(f"Queda média .............. {metrics.risco.average_drawdown(est.diario):.1%}")
print(f"Tempo abaixo do pico ..... {metrics.risco.time_under_water(est.diario):.0%} do período")

episodios = metrics.risco.episodios_drawdown(est.diario)
episodios.head(5)

**Profundidade e duração são riscos diferentes.** Uma queda de 15% que dura três anos
costuma ser mais difícil de aguentar que uma de 25% que dura dois meses — e só a segunda
aparece bem em qualquer ranking de performance.

## 5. Juntando as duas metades

Os ratios de risco-retorno existem para responder uma pergunta só: **quanto retorno por
unidade de incômodo?** O que muda entre eles é a definição de incômodo.

- **Sharpe** — incômodo é oscilação (para os dois lados)
- **Sortino** — incômodo é oscilação só para baixo
- **Calmar** — incômodo é a queda máxima

In [ ]:
metrics.risco.resumo(est.diario).round(3)

### A armadilha do Sharpe

Sharpe é tratado como nota de prova, e não é. Ele supõe que **volatilidade mede risco** —
o que deixa de valer quando a distribuição tem cauda esquerda pesada.

Sharpe alto com assimetria muito negativa é um alerta, não um elogio. Veremos como medir
isso no nível 2.

## 6. Comparar com o quê?

Um retorno sem referência não significa nada. O manual de marca é explícito: comparar
sempre **no mesmo período** e com referência adequada.

In [ ]:
metrics.execucao.beta_e_alfa(est.diario.values, bench.values).round(4)

O `beta` diz quanto do resultado é simplesmente estar comprado. O `excesso_por_periodo` é
o que sobra depois disso — e o `t_excesso` já antecipa a pergunta do nível 2: **esse
excesso é grande o suficiente frente ao erro de medição?**

## 7. A operação no tempo

O diagnóstico comportamental mais direto, e o que não precisa de estatística nenhuma.

In [ ]:
print("Trades por ano:", round(metrics.execucao.trades_por_ano(est.trades), 1))
print()
print(metrics.execucao.holding_period(est.trades).round(1).to_string())
print()
print(metrics.execucao.winner_vs_loser_holding(est.trades).round(1).to_string())

**Se as perdas ficassem abertas mais tempo que os ganhos**, o sistema estaria realizando
lucro cedo e carregando prejuízo esperando voltar. É o viés de disposição, visível em
duas linhas de código.

---

## O que levar deste nível

1. Win rate sozinho não diz nada. Leia sempre com payoff e break-even.
2. Média e mediana contam histórias diferentes sobre quem sustenta o resultado.
3. Profundidade e duração de queda são riscos distintos.
4. Sharpe supõe que volatilidade é risco. Nem sempre é.
5. Retorno sem benchmark e sem período declarado não é informação.

**No nível 2:** todos esses números são estimativas com incerteza em volta — e quase
nenhum sobrevive inteiro depois dos custos.

---

### Bloco de transparência

**Natureza:** educacional · **Dados:** simulados e reprodutíveis por semente ·
**Código:** aberto em [wizz-lab](https://github.com/Gustavofthiesen/wizz-lab)

Este material apresenta um processo de estudo, com finalidade educacional. Não
constitui recomendação individualizada, oferta ou promessa de retorno. Premissas podem
estar erradas e resultados passados não garantem resultados futuros.